## Phase 3: Training

In [1]:
from utils.pipeline import CustomPipeline, load_config
from utils.pipeline.config import SchedulerConfig

/home/alief/miniconda3/envs/kaggle/lib/python3.12/site-packages/trimesh/grouping.py:15: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.5.0)
  from scipy.spatial import cKDTree


In [2]:
cfg = load_config("configs/m3-xx.yaml")

cfg.data.max_samples = None
cfg.data.batch_size = 16
pipe = CustomPipeline(cfg)

In [3]:
# Consistency Check: DualSeq Float vs 8-Bit Tokenized Arguments
pipe.load_tokenizer()
pipe.load_dataset()
meta = pipe.metadata

Loading split from JSON: data/text2cad/train_test_val.json
Loading CSV from: /home/alief/prjkcad/data/text2cad/text2cad_v1.1.csv
Processing 118299 rows to load JSON payloads...


Loading Text2CAD JSONs: 100%|██████████| 118299/118299 [00:52<00:00, 2271.16it/s]


Processing 892 rows to load JSON payloads...


100%|██████████| 118299/118299 [00:08<00:00, 13408.99it/s]


Successfully created 118240/118299 DualSeq instances.


100%|██████████| 892/892 [00:00<00:00, 14321.22it/s]


Loading CSV from: /home/alief/prjkcad/data/text2cad/text2cad_v1.1.csv
Processing 8046 rows to load JSON payloads...


100%|██████████| 8046/8046 [00:00<00:00, 15979.16it/s]


Successfully created 8042/8046 DualSeq instances.


In [4]:
from utils.data_utils.dualseq_dataset import DualSeqDataset
from utils.representations.dual_seq import get_dualseq_schema
import pandas as pd
import numpy as np

dualseq_schema = get_dualseq_schema()

# Compare FloatArgs vs EightBitBinarizedArgs on dataset samples
sample_seqs = pipe.dual_seqs[:50]
tok = pipe.text_tokenizer if hasattr(pipe, "text_tokenizer") and pipe.text_tokenizer is not None else pipe.tokenizer
ds_float = DualSeqDataset(sample_seqs, tok, out_type="FloatArgs", metadata=meta)
ds_binned = DualSeqDataset(sample_seqs, tok, out_type="EightBitBinarizedArgs", metadata=meta)

records = []
for i in range(len(sample_seqs)):
    _, _, _, y_float = ds_float[i]
    _, _, _, y_binned = ds_binned[i]
    for t in range(y_float.shape[0]):
        cmd = sample_seqs[i].cmds[t]
        for arg_idx, arg_name in enumerate(dualseq_schema["arg_names"]):
            bin_tok = y_binned[t, arg_idx].item()
            if bin_tok != 256:  # non-padding token
                orig_val = y_float[t, arg_idx].item()
                recon_val = meta.bin_to_float(arg_name, bin_tok)
                diff = abs(orig_val - recon_val)
                step_size = (meta.max_vals[arg_name] - meta.min_vals[arg_name]) / 256.0
                records.append({
                    "Sample_ID": sample_seqs[i].uid,
                    "Cmd": cmd,
                    "Arg_Key": arg_name,
                    "Original_Float": orig_val,
                    "8Bit_Token": bin_tok,
                    "Recon_Float": recon_val,
                    "Abs_Error": diff,
                    "8Bit_Step_Size": step_size,
                    "Consistent": diff <= (step_size + 1e-4)
                })

df_check = pd.DataFrame(records)
all_consistent = df_check["Consistent"].all()
print(f"Consistency Check Result: {"PASSED (All Consistent)" if all_consistent else "FAILED"}")
print(f"Total Active Parameters Checked: {len(df_check)}")
print(f"Max Reconstruction Error: {df_check["Abs_Error"].max():.6f}")
print(f"Mean Reconstruction Error: {df_check["Abs_Error"].mean():.6f}")
display(df_check.head(10))

Consistency Check Result: PASSED (All Consistent)
Total Active Parameters Checked: 2294
Max Reconstruction Error: 0.703125
Mean Reconstruction Error: 0.047097


,Sample_ID,Cmd,Arg_Key,Original_Float,8Bit_Token,Recon_Float,Abs_Error,8Bit_Step_Size,Consistent
0,0035/00351230,COOR,coor_euax,0.00,128,0.703125,0.703125,1.406250,True
1,0035/00351230,COOR,coor_euay,0.00,129,0.069148,0.069148,0.343966,True
2,0035/00351230,COOR,coor_euaz,0.00,123,0.443350,0.443350,1.355145,True
3,0035/00351230,COOR,coor_tx,0.00,232,0.013947,0.013947,0.076913,True
4,0035/00351230,COOR,coor_ty,0.00,164,-0.007387,0.007387,0.030820,True
5,0035/00351230,COOR,coor_tz,0.00,214,-0.007335,0.007335,0.027827,True
6,0035/00351230,LINE,line_sx,0.00,0,0.009887,0.009887,0.019773,True
7,0035/00351230,LINE,line_sy,0.00,0,0.008789,0.008789,0.017578,True
8,0035/00351230,LINE,line_ex,0.75,37,0.741489,0.008511,0.019773,True
9,0035/00351230,LINE,line_ey,0.00,0,0.008789,0.008789,0.017578,True


In [5]:
pipe.load()

Loading CSV from: /home/alief/prjkcad/data/text2cad/text2cad_v1.1.csv
Processing 8046 rows to load JSON payloads...


100%|██████████| 8046/8046 [00:00<00:00, 17520.46it/s]


Successfully created 8042/8046 DualSeq instances.


The following layers were not sharded: embeddings.token_type_embeddings.weight, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.attention.self.key.bias, encoder.layer.*.attention.self.query.bias, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.attention.self.query.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.attention.self.value.bias, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.self.key.weight, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.attention.output.LayerNorm.bias, encoder.layer.*.attention.self.value.weight, embeddings.LayerNorm.weight, embeddings.word_embeddings.weight, embeddings.LayerNorm.bias, encoder.layer.*.output.dense.bias, pooler.dense.weight, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.attention.output.dense.bias, embeddings.position_embeddings.weight, pooler.dense.bias


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded checkpoint weights from: out/m3-xx/checkpoint.ckpt


In [6]:
pipe.run()

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 14421037 (14421037-institut-teknologi-bandung) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Training:   0%|          | 0/147800 [00:00<?, ?it/s]
  | Name      | Type            | Params | Mode 
------------------------------------------------------
0 | wrapper   | CustomWrapper   | 180 M  | train
1 | criterion | CustomCriterion | 0      | train
------------------------------------------------------
70.6 M    Trainable params
109 M     Non-trainable params
180 M     Total params
721.894   Total estimated model params size (MB)
405       Modules in train mode
228       Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Training:  15%|█▌        | 22170/147800 [1:23:02<7:48:04,  4.47it/s, global_step=22170, epoch=2]

...

Training:  90%|█████████ | 133020/147800 [8:46:10<54:55,  4.48it/s, global_step=133020, epoch=17]      

Validation: |          | 0/? [00:00<?, ?it/s]

Training: 100%|██████████| 147800/147800 [9:46:42<00:00,  4.20it/s, global_step=147800, epoch=19]


Saved final checkpoints to: out/m3-xx


[{'val/loss': 0.24933278560638428,
  'val/perplexity': 1.2834932804107666,
  'eval/LINE_precision': 0.5453153252601624,
  'eval/LINE_recall': 0.75,
  'eval/LINE_f1': 0.5889743566513062,
  'eval/CIRCLE_precision': 0.25,
  'eval/CIRCLE_recall': 0.2161458432674408,
  'eval/CIRCLE_f1': 0.22499999403953552,
  'eval/ARC_precision': 0.07848557829856873,
  'eval/ARC_recall': 0.125,
  'eval/ARC_f1': 0.09126462042331696,
  'eval/avg_f1': 0.30174630880355835,
  'eval/EXTRUDE_accuracy': 0.2682291567325592,
  'eval/arg_float_mse': 24295.0859375,
  'eval/arg_float_r2': -16.755155563354492,
  'eval/arg_token_accuracy': 0.0,
  'eval/arg_token_f1': 0.0,
  'eval/arg_sep_count_mse': 0.0,
  't2c/line_recall': 0.7025331258773804,
  't2c/line_precision': 0.5271965265274048,
  't2c/line_f1': 0.5636577606201172,
  't2c/line_correct_type': 5.875,
  't2c/line_total_pred': 14.96875,
  't2c/line_total_gt': 6.6875,
  't2c/line_param_s_x': 0.0,
  't2c/line_param_s_y': 0.0,
  't2c/line_param_e_x': 0.0,
  't2c/line_p

In [7]:
# res = pipe.eval()
# print(res)

In [8]:
ds = pipe.infer("A cylindrical object with a flat circular top and bottom. The top surface is slightly recessed compared to the bottom.")

In [9]:
print(ds)

cmd  | args                                                                                                                                                                                           
-----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
COOR | {'coor_euax': 0.703125, 'coor_euay': 0.06914843750000443, 'coor_euaz': 0.4433496093750193, 'coor_tx': 0.013947070312500642, 'coor_ty': -0.007387109374999312, 'coor_tz': -0.007334765625000017}
COOR | {'coor_euax': 0.703125, 'coor_euay': 0.06914843750000443, 'coor_euaz': 0.4433496093750193, 'coor_tx': 0.013947070312500642, 'coor_ty': -0.007387109374999312, 'coor_tz': -0.007334765625000017}
COOR | {'coor_euax': 0.703125, 'coor_euay': 0.06914843750000443, 'coor_euaz': 0.4433496093750193, 'coor_tx': 0.013947070312500642, 'coor_ty': -0.007387109374999312, 'coor_tz': -0.007334765625000017}
COOR 

In [10]:
ds = pipe.infer("A simpl cube")

In [11]:
print(ds)

cmd         | args                                                                                                                                                                                           
------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
COOR        | {'coor_euax': 0.703125, 'coor_euay': 0.06914843750000443, 'coor_euaz': 0.4433496093750193, 'coor_tx': 0.16777285156249988, 'coor_ty': -0.007387109374999312, 'coor_tz': -0.007334765625000017} 
COOR        | {'coor_euax': 0.703125, 'coor_euay': 0.06914843750000443, 'coor_euaz': 0.4433496093750193, 'coor_tx': 0.16777285156249988, 'coor_ty': -0.007387109374999312, 'coor_tz': -0.007334765625000017} 
COOR        | {'coor_euax': 0.703125, 'coor_euay': 0.06914843750000443, 'coor_euaz': 0.4433496093750193, 'coor_tx': 0.16777285156249988, 'coor_ty': -0.007387109374999312, 'coor